# 04 — Tournament simulation

> **Mirrors `src/worldcup/simulation/simulator.py` and `predictions.py`.**
> Readable walkthrough; the code is the source of truth. We run fewer simulations here (5,000) for speed — the package default is 10,000.

In [1]:
from worldcup.features.elo import build_current_ratings
from worldcup.models.poisson import train_poisson
from worldcup.features.build import build_features
from worldcup.data.load import load_results
from worldcup.simulation.simulator import run_simulation, load_wc2026_played

elo = build_current_ratings(save=False)
poisson = train_poisson(build_features(load_results()), save=False)
played = load_wc2026_played()
print(f"Conditioning on {len(played)} played WC2026 matches.")

Conditioning on 44 played WC2026 matches.


## Title odds — conditioned on results so far

Played matches are **fixed**; only the remaining games are simulated.

In [2]:
table = run_simulation(n_sims=5000, seed=42, elo=elo, poisson_bundle=poisson, played_matches=played)
show = table.head(12).copy()
for c in ["qualify", "round_of_16", "quarter_final", "semi_final", "final", "champion"]:
    show[c] = (show[c] * 100).round(1)
show

,team,qualify,round_of_16,quarter_final,semi_final,final,champion
0,Argentina,100.0,94.2,59.3,43.6,29.7,22.0
1,Spain,100.0,87.2,52.4,37.6,20.5,14.9
2,France,100.0,84.8,44.7,28.4,16.9,11.3
3,England,99.9,86.6,46.3,32.0,14.3,9.2
4,Brazil,100.0,72.5,45.7,31.4,17.6,6.7
5,Colombia,99.1,74.5,55.8,24.1,9.5,5.4
6,Germany,100.0,77.1,42.2,24.9,14.7,5.2
7,Netherlands,100.0,60.9,32.1,19.5,11.4,4.7
8,Mexico,100.0,67.8,47.2,25.6,13.9,4.5
9,Morocco,100.0,59.1,33.4,21.5,10.8,3.1


## Elimination — teams already (nearly) out

In [3]:
low = table.sort_values("qualify").head(8)[["team", "qualify"]].copy()
low["qualify"] = (low["qualify"] * 100).round(1)
low

,team,qualify
43,Tunisia,0.0
44,Jordan,0.0
34,Turkey,0.0
33,Haiti,0.0
45,Iraq,10.5
37,Czech Republic,12.9
39,Curaçao,13.5
36,South Africa,13.7


## Official bracket

32 qualifiers (12 winners + 12 runners-up + 8 best thirds) play a fixed left/right tree. `assign_thirds` maps the advancing third-placed teams to winner slots without a group rematch.

In [4]:
from worldcup.simulation.simulator import assign_thirds
assign_thirds(["A", "B", "C", "D", "G", "H", "K", "L"])  # {winner-slot: 3rd-place group}

{'A': 'B',
 'B': 'A',
 'C': 'D',
 'D': 'C',
 'G': 'H',
 'H': 'G',
 'K': 'L',
 'L': 'K'}